# Langfuse Prompt Management

Bu notebook, telco agent'larının system prompt'larını Langfuse'a yükler.

**Convention:** Langfuse prompt adı = `create_agent(name=...)` değeri.

| Agent | Langfuse Prompt Name |
|-------|---------------------|
| Main (supervisor) | `main_agent` |
| Subscription | `subscription_agent` |
| Billing | `billing_agent` |
| Technical | `technical_agent` |

## 1. Setup

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langfuse import Langfuse

# .env dosyasini proje kokunden yukle
env_path = Path("../.env")
load_dotenv(env_path)

client = Langfuse()
print(f"Langfuse host: {os.environ.get('LANGFUSE_HOST')}")
print(f"Public key:    {os.environ.get('LANGFUSE_PUBLIC_KEY', '')[:20]}...")

Langfuse host: https://cloud.langfuse.com
Public key:    pk-lf-43cc37ae-8f5a-...


## 2. Prompt Tanımları

Her prompt, ilgili agent dosyasındaki `system_prompt` ile aynı içeriğe sahip.
Langfuse'da değiştirdiğinizde agent runtime'da yeni versiyonu otomatik alır (cache TTL: 60s).

In [2]:
PROMPTS = {
    "main_agent": (
        "You are a customer support manager for a telecom operator.\n\n"
        "Route customer questions to the right specialist:\n"
        "- Subscription questions (plan info, upgrades, packages) → ask_subscription_specialist\n"
        "- Billing questions (invoices, charges, payments) → ask_billing_specialist\n"
        "- Technical issues (signal, connectivity, device problems) → ask_technical_specialist\n"
        "- Greetings and general questions → answer directly\n\n"
        "Be friendly, concise, and always synthesize specialist responses "
        "into a clear answer for the customer."
    ),
    "subscription_agent": (
        "You are a subscription and plan specialist for a telecom operator. "
        "Help customers check their current plan, search for better plans, "
        "compare options side by side, change plans, and add extra packages.\n\n"
        "Always mention contract end dates and any applicable fees when changing plans. "
        "Proactively suggest plan upgrades when usage patterns indicate frequent overages."
    ),
    "billing_agent": (
        "You are a billing specialist for a telecom operator. "
        "Help customers view invoices, understand charges, check payment history, "
        "and set up installment payment plans.\n\n"
        "When you notice data overages or extra charges that recur, ALWAYS use "
        "suggest_plan_change to recommend a more suitable plan to the customer. "
        "Be transparent about all charges and explain any fees clearly."
    ),
    "technical_agent": (
        "You are a technical support specialist for a telecom operator. "
        "Help customers diagnose connectivity issues, check network status in their area, "
        "verify device compatibility, and create trouble tickets for unresolved problems.\n\n"
        "Always run diagnostics before creating a trouble ticket. "
        "If diagnostics show normal results, guide the customer through basic troubleshooting "
        "(restart device, toggle airplane mode, reset network settings) before escalating."
    ),
}

print(f"{len(PROMPTS)} prompt tanimlanadi:")
for name in PROMPTS:
    print(f"  - {name}")

4 prompt tanimlanadi:
  - main_agent
  - subscription_agent
  - billing_agent
  - technical_agent


## 3. Prompt'ları Langfuse'a Yükle

`create_prompt` her çağrıda yeni bir **versiyon** oluşturur.
Aynı isimle tekrar çalıştırırsanız v2, v3... olarak versiyonlanır.

In [3]:
for name, prompt_text in PROMPTS.items():
    resp = client.create_prompt(
        name=name,
        prompt=prompt_text,
        type="text",
        labels=["production"],
    )
    print(f"[OK] {name} -> version {resp.version}")

client.flush()
print("\nTum promptlar yuklendi.")

[OK] main_agent -> version 1
[OK] subscription_agent -> version 1
[OK] billing_agent -> version 1
[OK] technical_agent -> version 1

Tum promptlar yuklendi.


## 4. Doğrulama — Yüklenen Prompt'ları Oku

Langfuse'dan geri çekerek doğrulayın.

In [4]:
for name in PROMPTS:
    prompt = client.get_prompt(name)
    compiled = prompt.compile()
    print(f"--- {name} (v{prompt.version}) ---")
    print(compiled[:120] + "..." if len(compiled) > 120 else compiled)
    print()

--- main_agent (v1) ---
You are a customer support manager for a telecom operator.

Route customer questions to the right specialist:
- Subscrip...

--- subscription_agent (v1) ---
You are a subscription and plan specialist for a telecom operator. Help customers check their current plan, search for b...

--- billing_agent (v1) ---
You are a billing specialist for a telecom operator. Help customers view invoices, understand charges, check payment his...

--- technical_agent (v1) ---
You are a technical support specialist for a telecom operator. Help customers diagnose connectivity issues, check networ...



## 5. Tek Bir Prompt'u Güncelle (opsiyonel)

Belirli bir agent'ın prompt'unu değiştirmek isterseniz sadece o hücreyi düzenleyip çalıştırın.

In [5]:
# Ornek: main_agent promptunu guncelle
# Asagidaki metni duzenleyin ve hucreyi calistirin.

UPDATED_PROMPT = """\
You are a customer support manager for a telecom operator.

Route customer questions to the right specialist:
- Subscription questions (plan info, upgrades, packages) → ask_subscription_specialist
- Billing questions (invoices, charges, payments) → ask_billing_specialist
- Technical issues (signal, connectivity, device problems) → ask_technical_specialist
- Greetings and general questions → answer directly

Be friendly, concise, and always synthesize specialist responses
into a clear answer for the customer.
"""

resp = client.create_prompt(
    name="main_agent",
    prompt=UPDATED_PROMPT.strip(),
    type="text",
    labels=["production"],
)
client.flush()
print(f"[OK] main_agent -> version {resp.version}")

[OK] main_agent -> version 2
